<a href="https://colab.research.google.com/github/renasyan/datmin-tubes-mlbb/blob/main/TUBES_DATMIN_MOBILE_LEGEND.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DATA UNDERSTANDING

Sebelum memulai analisis data, kita perlu mengimpor beberapa library yang akan digunakan dalam.Data understanding adalah proses mengenal isi dan struktur data. Kami menganalisis data hero, hasil pertandingan, dan statistik untuk memahami pola awal yang relevan

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
from ast import literal_eval
from itertools import combinations
from math import comb
from tqdm import tqdm

### ?? pd.read_csv() untuk membaca file CSV dan mengubahnya menjadi DataFrame

In [ ]:
from google.colab import files
uploaded = files.upload()

df = pd.read_csv('results.csv')

print(df.head())

### ?? df.describe() digunakan untuk menampilkan ringkasan statistik deskriptif dari data numerik dalam sebuah DataFrame.

In [ ]:
df.describe()

### ?? df.info() untuk melihat ringkasan struktur DataFrame

In [ ]:
df.info()

In [ ]:
df.isna().sum()

# **DATA EXPLORATION**

In [ ]:
numeric_cols = df.select_dtypes(include=['number'])
correlation_matrix = numeric_cols.corr()

plt.figure(figsize=(10, 6))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title("Heatmap - Correlation Matrix")
plt.show()

In [ ]:
sns.countplot(x ='match_result', data = df)
plt.show()

In [ ]:
from ast import literal_eval
df_check = df
df['left_heroes'] = df['left_heroes'].apply(literal_eval)
df['right_heroes'] = df['right_heroes'].apply(literal_eval)

all_heroes = df_check['left_heroes'].sum() + df['right_heroes'].sum()
hero_counts = pd.Series(all_heroes).value_counts()

print("Top 5 paling sering muncul:")
print(hero_counts.head())

print("\nBottom 5 paling sering muncul:")
print(hero_counts.tail())

# **DATA CLEANSING**

Data cleansing adalah proses membersihkan data dari nilai yang tidak valid, kosong, duplikat, atau tidak konsisten sebelum dilakukan analisis dengan tujuan agar data lebih rapih, akurat, dan siap digunakan dalam proses data mining

### ?? df.duplicated().sum() digunakan untuk menghitung jumlah baris duplikat dalam DataFrame

In [ ]:
# df.duplicated().sum()

In [ ]:
print(df.columns)

### ?? df.drop() digunakan untuk menghapus baris atau kolom dari sebuah DataFrame

In [ ]:
df=df.drop(columns=["Unnamed: 0","file", "player", "ss_type", "opening_failure", "battle_id", "right_medals", "left_medals", "left_scores",	"right_scores" ], axis=1)
df

pada tahap ini digunakan boxplot untuk menampilkan distribusi data numerik secara singkat, disini boxplot menunjukkan data match_duration, left_team_kills, dan right_team_kills

In [ ]:
def display_boxplot():
  numerical_cols = df.select_dtypes(include=['number'])
  plt.figure(figsize=(20, 20))

  for i, col in enumerate(numerical_cols, 1):
    plt.subplot(5, 4, i)
    sns.boxplot(x=df[col])
    plt.title(f"Boxplot untuk {col}")

  plt.tight_layout()
  plt.show()

display_boxplot()

IQR (Interquartile Range) adalah ukuran penyebaran data yang menunjukkan jarak antara kuartil pertama (Q1) dan kuartil ketiga (Q3), Outlier adalah nilai data yang jauh berbeda dari nilai-nilai lain dalam dataset yang bisa saja terjadi karena kesalahan pengukuran, data yang tidak biasa, atau variasi alami

In [ ]:
numerical_cols = df.select_dtypes(include=['number'])

outlier_index = []
for column in numerical_cols:
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1

    outliers = df.index[(df[column] < (Q1 - 1.5 * IQR))| (df[column] > (Q3 + 1.5 * IQR))]

    print(f"{column} : {len(outliers)}")
    outlier_index.extend(outliers)

outlier_index = sorted(set(outlier_index))
print(outlier_index)
df=df.drop(index=outlier_index)

In [ ]:
display_boxplot()

In [ ]:
df.head()

# DATA TRANSFORMATION

Data Transformation adalah proses mengubah data dari format atau struktur asli menjadi format lain yang lebih sesuai untuk analisis atau pemodelan

Encoding adalah proses mengubah data kategori (misalnya nama hero, status menang/kalah) menjadi format numerik agar bisa diproses oleh algoritma komputer, terutama machine learning.

In [ ]:
# df_backup_capek_run_ulang = df
# df = df_backup_capek_run_ulang

### ?? **Encoding Victory**

In [ ]:
df['match_result'].value_counts()

In [ ]:
df['match_result'] = df['match_result'].map({
    'Victory':0,
    'Defeat':1
})

df['match_result'].value_counts()

In [ ]:
df.isna().sum()

### ?? **Membuat Transaction Hero Pada Masing Masing Tim**

In [ ]:
from ast import literal_eval
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

In [ ]:
df

In [ ]:
df_flip = df.copy()
df_flip['left_heroes'], df_flip['right_heroes'] = df['right_heroes'], df['left_heroes']
df_flip['match_result'] = 1 - df['match_result']
df_all = pd.concat([df, df_flip], ignore_index=True)

In [ ]:
all_heroes = set()
for row in pd.concat([df_all['left_heroes'], df_all['right_heroes']]):
    all_heroes.update(row)
all_heroes = sorted(all_heroes)


In [ ]:
# def convert_to_items(row):
#     result = []
#     result += [f"L_{h}" for h in row['left_heroes']]
#     result += [f"R_{h}" for h in row['right_heroes']]
#     result += ['WIN_L'] if row['match_result'] == 1 else ['WIN_R']
#     return result

# df_all['items'] = df_all.apply(convert_to_items, axis=1)

In [ ]:
df=df.drop(columns=["left_heroes", "right_heroes" ], axis=1)

In [ ]:
print(df.columns.tolist())
df
df.info()

# **MODELING**

In [ ]:
df

In [ ]:
def encode_match(row):
    encoding = dict.fromkeys(['L_' + h for h in all_heroes] + ['R_' + h for h in all_heroes], 0)
    for h in row['left_heroes']:
        encoding['L_' + h] = 1
    for h in row['right_heroes']:
        encoding['R_' + h] = 1
    return pd.Series(encoding)

X = df_all.apply(encode_match, axis=1)
y = df_all['match_result']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import cross_val_score

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]
scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')

print("Accuracy:", accuracy_score(y_test, y_pred)*100, "%")
print("ROC AUC:", roc_auc_score(y_test, y_prob)*100, "%")
print("Cross-val Accuracy:", scores.mean()*100, "%")

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()
plt.show()


In [ ]:

enemy_heroes = ['Valir', 'Moskov', 'Melissa']
enemy_features = ['R_' + h for h in enemy_heroes]
our_hero_pool = ['L_' + h for h in all_heroes]


from tqdm import tqdm

# Batched input
input_data = []
combos = []

for combo in combinations(our_hero_pool, 3):
    input_vec = dict.fromkeys(X.columns, 0)

    # Tambahkan hero musuh
    for h in enemy_heroes:
        h_col = 'R_' + h
        if h_col in input_vec:
            input_vec[h_col] = 1

    # Tambahkan hero kita
    for h in combo:
        if h in input_vec:
            input_vec[h] = 1

    input_data.append(input_vec)
    combos.append(tuple(h[2:] for h in combo))

# DataFrame input
input_df = pd.DataFrame(input_data)

# Prediksi seluruh kombinasi sekaligus
probs = model.predict_proba(input_df)[:, 1]

# Gabungkan hasil
recommendations = list(zip(combos, probs))
top_recommendations = sorted(recommendations, key=lambda x: x[1], reverse=True)[:5]

# Tampilkan hasil
print("Top hero combos vs", enemy_heroes)
for combo, prob in top_recommendations:
    print(f"{combo}: win probability = {prob:.2f}")